# IMO Health — Diagnosis Specificity Agent

This notebook provides a **Diagnosis Specificity Agent** that refines broad diagnoses to their
most specific form by traversing the IMO Knowledge Graph’s refinement hierarchy.

## What Does This Agent Do?

Given a clinical note, this agent will:
1. **Extract** base diagnoses/conditions from the note (stripping qualifiers)
2. **Normalize** the selected problem via IMO Precision Normalize API
3. **Query refinements** from the KG — allowedRefinements grouped by type, laterality, severity, etc.
4. **Match refinements** to clinical evidence in the note
5. **Resolve** the most specific concept via nested `refinementNarrower` graph traversal
6. **Present** the refined diagnosis with ICD-10 codes and IMO Studio links

| Component | Technology |
|-----------|------------|
| LLM | AWS Bedrock / OpenAI / Anthropic / Azure OpenAI (configurable) |
| Tools | IMO Normalize API + KG GraphQL (allowedRefinements, refinementNarrower) |
| Agent | LangGraph ReAct Agent |
| Auth | OAuth2 client_credentials grant |

## Prerequisites

- `config.json` file with IMO API + LLM credentials (copy from `config.json.template`)
- For AWS Bedrock: AWS credentials (SageMaker execution role or explicit keys)
- For OpenAI/Anthropic/Azure: respective API keys in config.json
- Python 3.10+

## Step 1: Install Dependencies

Run this cell once, then restart the kernel.

Installs support for all LLM providers. You only need credentials for the one you choose.

In [ ]:
%pip install -q --upgrade \
    "langchain-core>=1.0.0" \
    "langchain>=1.0.0" \
    "langchain-aws>=0.2.0" \
    "langchain-openai>=0.3.0" \
    "langchain-anthropic>=0.3.0" \
    "langgraph>=0.2.0" \
    boto3 botocore requests nest_asyncio python-dotenv \
    ipywidgets markdown

## Step 2: Configuration

Loads credentials from `config.json`. Create one from `config.json.template` if it doesn’t exist.

### LLM Provider

Set `llm.provider` in `config.json` to one of:
- `"bedrock"` — AWS Bedrock (Claude via AWS)
- `"openai"` — OpenAI (GPT-4o, etc.)
- `"anthropic"` — Anthropic direct API (Claude)
- `"azure_openai"` — Azure OpenAI Service

In [ ]:
import os
import sys
import json
import pathlib
import nest_asyncio

nest_asyncio.apply()

# --- Ensure notebook directory is on path for local imports ---
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# --- Load config.json ---
candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json',
]
cfg_path = next((p for p in candidates if p.exists()), None)
if cfg_path is None:
    raise FileNotFoundError(
        'config.json not found. Copy config.json.template to config.json and fill in your credentials.'
    )

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

# --- IMO API credentials ---
imo_cfg = cfg.get('imo_api', {})
imo_kg_cfg = cfg.get('imo_kg', {})

os.environ['IMO_NORMALIZE_CLIENT_ID'] = imo_cfg.get('client_id', '')
os.environ['IMO_NORMALIZE_SECRET'] = imo_cfg.get('client_secret', '')
os.environ['IMO_KG_CLIENT_ID'] = imo_kg_cfg.get('client_id', '') or imo_cfg.get('client_id', '')
os.environ['IMO_KG_CLIENT_SECRET'] = imo_kg_cfg.get('client_secret', '') or imo_cfg.get('client_secret', '')

# --- LLM configuration ---
llm_cfg = cfg.get('llm', {})
LLM_PROVIDER = llm_cfg.get('provider', 'bedrock')

# --- AWS credentials from config.json (for Bedrock) ---
aws_cfg = cfg.get('aws', {})
if aws_cfg.get('access_key_id'):
    os.environ.pop('AWS_PROFILE', None)
    os.environ['AWS_ACCESS_KEY_ID'] = aws_cfg['access_key_id']
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_cfg['secret_access_key']
    os.environ['AWS_SESSION_TOKEN'] = aws_cfg.get('session_token', '')
    os.environ['AWS_DEFAULT_REGION'] = aws_cfg.get('region', 'us-east-1')

print(f'Config loaded from : {cfg_path.resolve()}')
print(f'LLM Provider       : {LLM_PROVIDER}')
print(f'Normalize URL      : {imo_cfg.get("normalize_url", "https://api.imohealth.com/precision/normalize")}')
print(f'KG GraphQL URL     : {imo_cfg.get("graphql_url", "https://api.imohealth.com/knowledgegraph/graphql/")}')

## Step 3: Initialize LLM

Creates the LLM client based on the configured provider.  
Temperature is set to 0 for deterministic outputs across all providers.

In [ ]:
def create_llm(provider: str, llm_cfg: dict):
    """Create LLM instance based on the configured provider."""

    if provider == 'bedrock':
        from langchain_aws import ChatBedrockConverse
        bedrock_cfg = llm_cfg.get('bedrock', {})
        model_id = bedrock_cfg.get('model_id', 'us.anthropic.claude-haiku-4-5-20251001-v1:0')
        region = bedrock_cfg.get('region', 'us-east-1')
        llm = ChatBedrockConverse(
            model_id=model_id,
            region_name=region,
            temperature=0,
            max_tokens=64000,
            provider="anthropic",
        )
        print(f'LLM ready: AWS Bedrock ({model_id}, {region})')
        return llm

    elif provider == 'openai':
        from langchain_openai import ChatOpenAI
        openai_cfg = llm_cfg.get('openai', {})
        model = openai_cfg.get('model', 'gpt-4o')
        llm = ChatOpenAI(
            model=model,
            api_key=openai_cfg.get('api_key', ''),
            temperature=0,
        )
        print(f'LLM ready: OpenAI ({model})')
        return llm

    elif provider == 'anthropic':
        from langchain_anthropic import ChatAnthropic
        anthropic_cfg = llm_cfg.get('anthropic', {})
        model = anthropic_cfg.get('model', 'claude-sonnet-4-20250514')
        llm = ChatAnthropic(
            model=model,
            api_key=anthropic_cfg.get('api_key', ''),
            temperature=0,
            max_tokens=64000,
        )
        print(f'LLM ready: Anthropic ({model})')
        return llm

    elif provider == 'azure_openai':
        from langchain_openai import AzureChatOpenAI
        azure_cfg = llm_cfg.get('azure_openai', {})
        llm = AzureChatOpenAI(
            azure_deployment=azure_cfg.get('deployment', ''),
            azure_endpoint=azure_cfg.get('endpoint', ''),
            api_key=azure_cfg.get('api_key', ''),
            api_version=azure_cfg.get('api_version', '2024-02-15-preview'),
            temperature=0,
        )
        print(f'LLM ready: Azure OpenAI ({azure_cfg.get("deployment", "")})')
        return llm

    else:
        raise ValueError(
            f'Unknown LLM provider: "{provider}". '
            f'Supported: bedrock, openai, anthropic, azure_openai'
        )


llm = create_llm(LLM_PROVIDER, llm_cfg)

## Step 4: Define Agent Tools

Four tools power the diagnosis specificity workflow:

| Tool | Purpose |
|------|---------|
| `normalize_medical_term` | Normalize a diagnosis (domain=Problem) via IMO Precision Normalize API |
| `get_lexical` | Query KG for concept data: allowedRefinements, mappings, narrower subtypes |
| `get_allowed_refinements` | Get refinement groups (type, laterality, severity, chronicity) for a concept |
| `get_narrower_sequential_refinements` | Apply refinements sequentially via nested `refinementNarrower` traversal |

In [ ]:
from typing import List
from langchain_core.tools import tool
from kg_api_client import KGApiClient

_kg_client = KGApiClient()


@tool
def normalize_medical_term(input_term: str, domain: str = "Problem") -> str:
    """Normalize a medical term using IMO Precision Normalize API.

    Args:
        input_term: The clinical term (e.g., "Hypertension", "chest pain"). Must not be empty.
        domain: "Problem" for diagnoses, "Medication" for drugs, "Procedure" for procedures.

    Returns the normalized concept with:
    - title: canonical name
    - default_lexical_code: the stable IMO identifier used for KG lookups
    - score: confidence score
    - icd10_codes: mapped ICD-10-CM codes
    """
    if not input_term or not input_term.strip():
        return json.dumps({"success": False, "error": "STOP: input_term is empty. Do NOT retry with empty string."})
    result = _kg_client.normalize_medical_term(input_term.strip(), domain)
    return json.dumps(result, indent=2)


@tool
def get_lexical(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Query the IMO Knowledge Graph for a concept's core data.

    For problem domain: returns allowedRefinements, narrower subtypes, refinementFamilies, mappings.
    Large result lists (500+ items) are automatically truncated.

    Args:
        imo_lexical_code: The default_lexical_code from normalize results. REQUIRED.
        domain: "problem" or "medication"
    """
    if not imo_lexical_code or not imo_lexical_code.strip():
        return json.dumps({"success": False, "error": "STOP: imo_lexical_code is empty. Provide a valid lexical code."})
    result = _kg_client.get_lexical(imo_lexical_code.strip(), domain)
    return json.dumps(result, indent=2)


@tool
def get_allowed_refinements(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Get allowed refinements for a problem concept. Returns refinement groups
    (type, laterality, severity, chronicity, etc.) that can further narrow a diagnosis.

    Args:
        imo_lexical_code: The IMO lexical code of the problem
        domain: Use "Problem".
    """
    result = _kg_client.get_allowed_refinements(imo_lexical_code, domain)
    return json.dumps(result, indent=2)


@tool
def get_narrower_sequential_refinements(
    imo_lexical_code: str,
    refinement_sequence: List[List[str]],
    include_mappings: bool = True,
    domain: str = "Problem",
) -> str:
    """Apply refinements sequentially to resolve the most specific concept.

    Uses nested refinementNarrower calls — each step narrows from the previous result.
    Pass a MAXIMUM of 3 refinements per call. Chain multiple calls for more.

    Args:
        imo_lexical_code: Base concept lexical code
        refinement_sequence: Array of arrays, each inner array is one refinement step.
            Example: [["230"], ["362636588"]]
        include_mappings: Include ICD-10 mappings in response. ALWAYS set to false.
        domain: Use "Problem".
    """
    result = _kg_client.get_narrower_sequential_refinements(
        imo_lexical_code, refinement_sequence, include_mappings, domain
    )
    return json.dumps(result, indent=2)


tools = [normalize_medical_term, get_lexical, get_allowed_refinements, get_narrower_sequential_refinements]
print(f'Tools ready: {[t.name for t in tools]}')

## Step 5: System Prompt & Create Agent

The agent follows a 4-phase workflow:

1. **Phase 1 (Extract)** — Extract base diagnoses from the clinical note (text only, no tool calls), ask user which to refine
2. **Phase 2 (Normalize)** — Normalize the selected problem via IMO Normalize API
3. **Phase 3 (Refinements)** — Query KG for allowedRefinements, get refinement groups
4. **Phase 4 (Reason & Recommend)** — Match refinements to clinical evidence, resolve via `refinementNarrower`, present with ICD-10 codes

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a clinical diagnostic specificity agent that uses the IMO Normalize API and Knowledge Graph to find the most specific diagnosis for clinical encounters.

Your greeting (already shown to the user):
"I am a clinical diagnostic specificity agent that can find the most specific diagnosis for a patient visit. Provide the patient visit summary or clinical note and I will use IMO Normalize API and Knowledge Graph to get the highest level of diagnostic specificity."

## CRITICAL RULES — READ THESE FIRST
1. **Extraction Rule:** When a user sends a clinical note or encounter summary, you MUST respond with ONLY a text message listing extracted base problems. You are STRICTLY FORBIDDEN from calling any tools (normalize_medical_term, get_lexical, get_narrower_sequential_refinements) until the user replies and tells you which finding to refine. The ONLY exception: the user explicitly names a single diagnosis to refine (e.g., "refine hypertension"). A clinical note that contains an assessment is NOT a refinement request — it still requires extraction first.

2. **Tool Explanation Rule:** BEFORE calling ANY tool, you MUST output a brief explanation (1-2 sentences) of WHY you are calling that tool. This helps users understand your reasoning process. Never call a tool without first explaining what you're about to do and why.

3. **CRITICAL: Always Use default_lexical_code:** When calling get_lexical, you MUST use the `default_lexical_code` field from the normalize results, NOT the `lexical_code` field. The normalize API returns both fields, but `default_lexical_code` is the canonical code that must be used for Knowledge Graph lookups.

## Your Workflow

### Phase 1: Extract the Base Problem (TEXT ONLY — NO TOOL CALLS)
When the user provides a clinical note or encounter summary:

**Case A — User explicitly names a diagnosis to refine** (e.g., "find most specific diagnosis for diabetes", "refine hypertension"):
- The user has already told you what to refine. Do NOT list findings or ask.
- **Skip directly to Phase 2** using the diagnosis the user mentioned.
- Proceed through Phase 2 → 3 → 4 in one continuous flow.

**Case B — User provides a clinical note (THIS IS THE DEFAULT CASE):**
- Read the **entire** note carefully — look at History, HPI, Assessment, and Plan sections
- Identify all **diagnoses and conditions** mentioned anywhere in the note (not just the Assessment). Do NOT extract individual symptoms, signs, vitals, or lab values (e.g., do NOT extract "productive cough", "fever", "crackles", "elevated WBC" as separate findings).
- Extract ONLY the **base diagnoses/conditions** — strip away ALL qualifiers such as severity (e.g., "sharp", "acute"), laterality (e.g., "left-sided", "right", "bilateral"), anatomical location details (e.g., "right lower lobe"), chronicity ("chronic", "recurrent"), typing (e.g., "type 2"), and associated conditions.
- Examples of correct extraction:
  - Note: "Patient presents with sharp, left-sided chest pain... Assessment: Chest pain, likely musculoskeletal vs anginal" → extract **"Chest pain"**
  - Note: "...Assessment: Type 2 diabetes with diabetic peripheral neuropathy" → extract **"Diabetes mellitus"** (NOT "Type 2 diabetes", NOT "tingling in feet")
  - Note: "65-year-old female with essential hypertension and chronic kidney disease stage 3... Assessment: Hypertensive chronic kidney disease with stage 3 CKD" → extract **"Hypertension"** and **"Chronic kidney disease"** (NOT "hypertensive chronic kidney disease")
  - Note: "...Assessment: Community-acquired pneumonia, right lower lobe" → extract **"Pneumonia"** (NOT "community-acquired pneumonia, right lower lobe", NOT "productive cough", NOT "fever")
  - Note: "Nasal congestion from fall pollen and allergies" → extract **"Nasal congestion"** (NOT "Nasal congestion (from fall pollen and allergies)")
- **CRITICAL OUTPUT FORMAT:** Present a numbered summary of the extracted BASE diagnoses to the user. List ONLY the base diagnosis names with NO additional text, NO qualifiers, NO explanations, and NO parenthetical information whatsoever.
  - ✓ CORRECT: "1. Nasal congestion"
  - ✗ WRONG: "1. Nasal congestion (from fall pollen and allergies)"
  - ✗ WRONG: "1. Congestive heart failure (CHF)"
  - ✗ WRONG: "1. Depression (patient reports feeling down)"
- Ask: "Please specify which finding(s) you would like to refine."
- **YOU MUST STOP HERE. DO NOT CALL ANY TOOLS. WAIT FOR THE USER TO REPLY.**

### Phase 2: Normalize & Identify (IMO Normalize API)
1. **BEFORE calling the normalize tool**, output a brief explanation (1-2 sentences) of why you need to call it
   - Example: "I will normalize the term 'Hypertension' using the IMO Normalize API to get the lexical code for Knowledge Graph lookup."
   - Example: "I will normalize the term 'Chest pain' to retrieve the standardized lexical code needed to query refinements."
2. Use **normalize_medical_term** to normalize the base problem via the IMO Precision Normalize API
   - Use domain "Problem" for diagnoses/conditions
   - Pass the term as a string in the `input_term` parameter (e.g., `input_term: "chest pain"`)
   - Use the `domain` parameter with value "Problem", "Procedure", "Medication", or "Lab"
   - Normalize the BASE term only (e.g., "chest pain", "diabetes mellitus", "hypertension", "pneumonia")
3. From the normalize results, identify the best matching base result (highest score, most relevant title and ICD-10 codes)

### Phase 3: Query Refinements via Knowledge Graph
3. **BEFORE calling the graphql tool**, output a brief explanation (1-2 sentences) of why you need to call it
   - Example: "Now retrieving refinements from the Knowledge Graph to identify available hypertension types and subtypes."
   - Example: "Now querying the Knowledge Graph using lexical code 29688 to retrieve all available refinement groups for diabetes mellitus."
4. Use **get_lexical** with the **default_lexical_code** from the best normalize result to retrieve refinements
   - **CRITICAL:** You MUST use the `default_lexical_code` field from the normalize response, NOT `lexical_code`
   - The normalize API returns BOTH fields: `lexical_code` and `default_lexical_code`
   - Example from normalize response: `{"title": "Hypertension", "lexical_code": "86491", "default_lexical_code": "86491", ...}`
   - **Always extract and use the `default_lexical_code` value** (e.g., `"default_lexical_code": "86491"`)
   - Pass the `default_lexical_code` value as the `imo_lexical_code` parameter to get_lexical
5. The allowed refinements returned by get_lexical are already grouped by category — use them directly to analyze which refinements match the clinical note
6. Do NOT simply dump the raw refinement data to the user — you must analyze it internally

### Phase 4: Reason & Recommend (THIS IS THE MOST IMPORTANT PHASE)
After gathering all refinement data, you MUST do the following:

**Step 1 — Match refinements to clinical evidence:**
For EACH refinement group, review the original clinical note and determine:
- Does the note contain words, phrases, or clinical findings that support a specific refinement? If yes, select it and cite the supporting evidence.
- If no refinement in the group has supporting evidence in the note, mark it as an "evidence gap". Do NOT select any refinement for that group.

**Refinement selection rules:**
- Select a refinement when the note provides supporting clinical evidence — this can be a direct quote OR a clear clinical implication from documented findings.
- Examples of valid clinical implications:
  - "sharp chest pain worse with exertion" → supports **"Acute"** (onset pattern implies acute presentation)
  - "left-sided chest pain" → supports **"Left"** laterality
  - "chronic kidney disease stage 3, eGFR 45" in a hypertension patient → supports **"secondary to renal failure"** (CKD is the underlying renal condition)
  - "right lower lobe infiltrate" → supports **"Right"** laterality and **"Lower lobe of lung"** location
  - "tingling in both feet, decreased sensation" in a diabetes patient → supports **"with peripheral neuropathy"**
- NEVER select "unspecified", "other", "NOS", or catch-all/default refinements. If the note doesn't provide enough info to pick a specific refinement, it's an evidence gap.
- NEVER fabricate clinical details not supported by the note. The note must contain SOME evidence (direct or implied) for the refinement.
- Do NOT add refinements that contradict or go beyond what the note describes.

**Step 2 — Build the most specific diagnosis(es):**
Combine the base diagnosis with selected refinements to form the most specific diagnosis possible for each valid path. The base diagnosis MUST always be included in each final combined name. For example, if the base is "Chest pain" and the selected refinement is "due to unstable angina pectoris", the most specific diagnosis is "Chest pain due to unstable angina pectoris" — NOT just "Unstable angina pectoris". The refinement refines the base; it does not replace it.

**Step 2.5 — Resolve the FINAL concept using graph traversal (MANDATORY):**
- You MUST use **get_narrower_sequential_refinements** to resolve the final concept.
- This tool performs NESTED/CHAINED traversal: it applies refinements sequentially, where each refinement narrows down from the results of the previous refinement.
- IMPORTANT: Due to graph depth limits, pass a MAXIMUM of 3 refinements per call. If you have more than 3 selected refinements, you MUST chain multiple calls:
  1. Order all selected refinements by clinical hierarchy: type → complication → laterality → severity → other qualifiers
  2. Take the first 3 refinements and call the tool with the base `imo_lexical_code`
  3. From the response, extract the `code` of the best resolved concept
  4. Use that resolved `code` as the new `imo_lexical_code` for the next call with the next batch of up to 3 refinements
  5. Repeat until all refinements are applied
  6. The final resolved concept from the last call is the answer
- Tool parameters:
  - `imo_lexical_code` (string): The base concept code to start from (or the resolved code from a previous call)
  - `refinement_sequence` (array of arrays): Up to 3 refinements per call, where each inner array contains one refinement code per step
  - `include_mappings` (boolean): **ALWAYS set to false** — the server has a known schema issue with mappings. Get mappings separately via normalize after resolving the concept.
- Example with 5 refinements (chunked into 2 calls):
  Call 1:
  ```
  {
    "imo_lexical_code": "601056",
    "refinement_sequence": [["230"], ["362636588"], ["1328819463"]],
    "include_mappings": false
  }
  ```
  → Returns concept with code "789012"
  Call 2:
  ```
  {
    "imo_lexical_code": "789012",
    "refinement_sequence": [["6840"], ["2126"]],
    "include_mappings": false
  }
  ```
  → Returns final resolved concept
- **CRITICAL: Always set `include_mappings: false`** on every call. To get ICD-10 codes for the final resolved concept, use normalize_medical_term with the resolved title instead.
- The agent should determine the optimal order for applying refinements based on clinical logic and refinement group hierarchy (e.g., type → complication → laterality → severity).
- If you have conflicting options within the same refinement group (e.g., "right" vs "left" laterality), you may make separate calls to explore different paths. But all non-conflicting refinements MUST be included in the same chain.
- If any intermediate call returns an empty `narrower` array (no match), try reordering the refinements or dropping the last refinement in that batch and retrying.
- Never skip this step. Never pick the final concept by text-only composition when this tool can be used.

**Step 3 — Verify the refined diagnosis codes:**
BEFORE calling the normalize tool for verification, output a brief explanation:
- Example: "Analyzing refinements against the clinical note. Now verifying the refined diagnosis code:"
- Example: "Now verifying the final refined diagnosis to retrieve the standardized ICD-10 code:"
Then use **normalize_medical_term** to normalize each final graph-resolved diagnosis from Step 2.5 (pass as `input_term: "<diagnosis>"`, `domain: "Problem"`).
This verification normalization retrieves the standardized title and ICD-10 code for the refined diagnosis. Do NOT reuse the base diagnosis codes — they are for the unmodified term only. Do NOT invent or guess codes.

**Step 4 — Present the final recommendation in this EXACT format:**

#### Refinement Analysis
| Refinement Group | Selected Refinement | Evidence from Note | Reasoning |
|---|---|---|---|
| (group name) | (chosen refinement or "No evidence") | (quote from note) | (why selected/rejected) |

#### Final Recommendation
- **Base Diagnosis:** (name) — ICD-10: (code from normalize results)
- **Selected Refinements:** (list each refinement selected with evidence)
- **Most Specific Diagnosis:** (the full refined diagnosis — MUST include the base diagnosis name combined with all selected refinements, e.g., "Chest pain due to unstable angina pectoris", NOT just the refinement alone) — ICD-10: (code from verification normalization)
- **Evidence Gaps (informational only):** The Knowledge Graph has refinements for these categories, but the clinical note does not contain evidence to support them. These do NOT mean the diagnosis is incomplete.

If one or more tested combinations resolve to concepts, provide:
- **Most Specific Diagnoses (Resolved Combination Paths):** list EACH resolved path separately, including the exact selected refinements for that path, resolved concept title/code, and verified ICD-10 code.

For each resolved path, you MUST use this path format:
- `Path N — <focus label>:`
- `Refined Diagnosis: <resolved diagnosis title>`
- `IMO Lexical Code: <lexical code>`
- `IMO KG Link: https://studio.imohealth.com/#/terminology-browser/graph?id=<lexical code>`
- `ICD-10 Codes:`
    - `Primary: <first ICD-10 code>`
    - `Secondary: <second ICD-10 code>` (if present)
    - `Tertiary: <third ICD-10 code>` (if present)
    - `Additional: <remaining ICD-10 codes comma-separated>` (if more than 3)
- `Evidence: <concise supporting evidence from note>`

ICD-10 ordering and highlighting rules:
- When multiple ICD-10 codes exist for a term, always label the first as **Primary**, second as **Secondary**, and third as **Tertiary**.
- Do not collapse multi-code outputs into a single unlabeled ICD line.
- If only one ICD-10 code exists, show Primary only.

## Important Rules
- NEVER call any tools when the user first provides a clinical note — ALWAYS extract base problems and ask first
- In Phase 1, ALWAYS extract only the base problem — never include severity, laterality, anatomical details, or typing qualifiers in the extracted finding
- Specificity comes from REFINEMENTS discovered via the Knowledge Graph, not from the initial normalization
- ONLY select a refinement when there is supporting evidence (direct quote or clear clinical implication) from the note — never select "unspecified", "other", or "NOS" refinements
- Do NOT just list raw refinement groups to the user — always analyze them against the note
- The goal is to return the most specific diagnosis set supported by evidence and graph resolution: include every valid tested combination path that resolves to a concept.
- If the note lacks information for a refinement group, leave it out of the diagnosis entirely and list it as an "evidence gap"
- Do NOT guess or hallucinate clinical information not present in the note
- NEVER fabricate ICD-10 codes — always get them from normalize tool results
- CRITICAL: You MUST ALWAYS call the tools (normalize_medical_term, get_lexical, and get_narrower_sequential_refinements) for EVERY refinement request, even if you have seen similar data in previous conversation turns. NEVER skip tool calls or reuse results from memory — always make fresh API calls. Each refinement MUST follow Phase 2 → 3 → 4 with actual tool calls."""


agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print('Diagnosis Specificity Agent ready.')

## Step 6: Clinical Note

The following clinical note will be used to test the agent. It contains diagnoses
that the agent will extract and refine through the KG hierarchy.

In [ ]:
CLINICAL_NOTE = """65-year-old female with essential hypertension and chronic kidney disease stage 3. BP today 148/92. eGFR 45. Proteinuria present on urine dipstick. Assessment: Hypertensive chronic kidney disease with stage 3 CKD. Increase lisinopril, recheck labs in 3 months."""

print("Clinical note loaded (Hypertension + CKD Stage 3).")
print(f"Length: {len(CLINICAL_NOTE)} characters")

## Step 7: Run the Agent

Send the clinical note to the agent. The agent will:
1. Extract base diagnoses and ask which to refine
2. On your reply, normalize → query refinements → resolve → recommend

This cell runs the initial extraction. Use Step 8 (Interactive Chat) to
continue the multi-turn conversation.

## Step 8: Interactive Chat Mode

Multi-turn interactive chat for the Diagnosis Specificity Agent.  
The agent extracts base problems, then you choose which to refine.

- Paste a clinical note → agent extracts findings
- Reply with which finding to refine (e.g., `Refine: Hypertension`)
- Agent normalizes → queries KG refinements → resolves → recommends

Commands: `quit` to end, `reset` to clear history.

In [ ]:
import asyncio
import json
import html as html_module
from IPython.display import display, HTML, Markdown


class AgentUI:
    """Rich HTML display for agent streaming output."""

    @staticmethod
    def header():
        display(HTML("""
        <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    border-radius:12px; color:white; margin-bottom:16px;">
            <h2 style="margin:0;">Diagnosis Specificity Agent</h2>
            <p style="margin:4px 0 0 0; opacity:0.9;">Clinical Note &rarr; Normalize &rarr; KG Refinements &rarr; Most Specific Diagnosis</p>
            <p style="margin:4px 0 0 0; opacity:0.7; font-size:12px;">Type <b>quit</b> to end &middot; <b>reset</b> to clear history &middot; <b>Kernel Interrupt</b> to stop mid-generation</p>
        </div>
        """))

    @staticmethod
    def status(message):
        display(HTML(f'<div style="color:#5f6368; font-size:12px; font-style:italic; padding:4px 0;">{html_module.escape(message)}</div>'))

    @staticmethod
    def reasoning(text):
        clean = html_module.escape(text.strip())
        display(HTML(f"""
        <div style="background:#fff8e1; border-left:4px solid #f59e0b; border-radius:0 8px 8px 0; padding:10px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#d97706;">&#128161; Agent Reasoning</b>
            <div style="margin-top:6px; color:#92400e; white-space:pre-wrap; max-height:200px; overflow-y:auto;">{clean}</div>
        </div>
        """))

    @staticmethod
    def tool_call(tool_name, parameters):
        params_json = html_module.escape(json.dumps(parameters, indent=2, default=str))
        display(HTML(f"""
        <div style="background:#e8f0fe; border:1px solid #1a73e8; border-radius:8px; padding:12px 16px; margin:8px 0; font-family:monospace; font-size:13px;">
            <b style="color:#1a73e8;">&#128295; Tool Call: {html_module.escape(tool_name)}</b>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:12px; overflow-x:auto;">{params_json}</pre>
        </div>
        """))

    @staticmethod
    def tool_result(tool_name, result):
        result_str = str(result)
        preview = html_module.escape(result_str[:400])
        full = html_module.escape(result_str)
        char_count = len(result_str)
        display(HTML(f"""
        <div style="background:#e6f4ea; border:1px solid #137333; border-radius:8px; padding:12px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#137333;">&#9989; Result: {html_module.escape(tool_name)}</b>
            <details style="margin-top:8px;">
                <summary style="cursor:pointer; font-weight:600; font-size:12px; color:#137333;">View full result ({char_count:,} chars)</summary>
                <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:11px; max-height:300px; overflow:auto;">{full}</pre>
            </details>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:4px; font-size:11px; opacity:0.7;">{preview}{'...' if char_count > 400 else ''}</pre>
        </div>
        """))

    @staticmethod
    def agent_response(text):
        display(Markdown(text))

    @staticmethod
    def separator():
        display(HTML('<hr style="border:none; border-top:2px solid #dadce0; margin:16px 0;">'))

    @staticmethod
    def stopped():
        display(HTML("""
        <div style="background:#fce8e6; border:1px solid #c5221f; border-radius:8px; padding:12px 16px; margin:8px 0; text-align:center;">
            <b style="color:#c5221f;">Session ended.</b>
        </div>
        """))


ui = AgentUI()
chat_messages = []
ui.header()

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() == 'quit':
        ui.stopped()
        break
    if user_input.lower() == 'reset':
        chat_messages = []
        ui.status('Conversation reset. Paste a new clinical note.')
        continue

    display(HTML(f"""
    <div style="background:#f0f0f0; border-radius:8px; padding:10px 16px; margin:8px 0; font-size:14px;">
        <b>You:</b> {html_module.escape(user_input[:500])}{'...' if len(user_input) > 500 else ''}
    </div>
    """))

    chat_messages.append({'role': 'user', 'content': user_input})
    ui.status('Agent is thinking...')

    final_content = ""
    last_reasoning = ""

    try:
        async for chunk in agent.astream(
            {'messages': chat_messages},
            stream_mode='updates'
        ):
            for node_name, node_output in chunk.items():
                if node_name == 'agent':
                    msgs = node_output.get('messages', [])
                    for msg in msgs:
                        if hasattr(msg, 'content') and msg.content:
                            if isinstance(msg.content, str) and msg.content:
                                final_content = msg.content
                                last_reasoning = msg.content
                            elif isinstance(msg.content, list):
                                text_parts = []
                                for block in msg.content:
                                    if isinstance(block, dict) and block.get('type') == 'text':
                                        text_parts.append(block['text'])
                                    elif isinstance(block, str):
                                        text_parts.append(block)
                                if text_parts:
                                    joined = ''.join(text_parts)
                                    final_content = joined
                                    last_reasoning = joined

                        if hasattr(msg, 'tool_calls') and msg.tool_calls:
                            if last_reasoning:
                                ui.reasoning(last_reasoning)
                                last_reasoning = ""
                            for tc in msg.tool_calls:
                                ui.tool_call(tc['name'], tc.get('args', {}))

                elif node_name == 'tools':
                    msgs = node_output.get('messages', [])
                    for msg in msgs:
                        if hasattr(msg, 'content'):
                            ui.tool_result(
                                getattr(msg, 'name', 'tool'),
                                msg.content
                            )

    except KeyboardInterrupt:
        print('Generation interrupted.')

    if final_content:
        ui.separator()
        ui.agent_response(final_content)
        chat_messages.append({'role': 'assistant', 'content': final_content})

    ui.separator()
